In [1]:
import sys
!{sys.executable} -m pip install --quiet torch torchvision torchaudio
import torch
!pip install xgboost
!pip install optuna
!pip install --upgrade category_encoders
!pip install pandas
!pip install optuna optuna-dashboard
!pip install hyperopt
import os
import pandas as pd
import numpy as np
import csv
import sklearn
import ast
import xgboost as xgb
import category_encoders as catenc
import optuna
from optuna.pruners import ThresholdPruner
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression, ElasticNet
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder, OrdinalEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, confusion_matrix, mean_squared_error, r2_score, classification_report, roc_auc_score, mean_absolute_error, root_mean_squared_error
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from category_encoders.target_encoder import TargetEncoder
from sklearn.model_selection import KFold
from sklearn.preprocessing import MultiLabelBinarizer
from IPython.display import Audio

trainData = pd.read_csv(r"train.csv")
CompData = pd.read_csv(r"test.csv")

print("Setup Complete")


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


c:\Users\sbelu\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setup Complete


In [ ]:
#X and y seperation
y = trainData["accident_risk"]
X = trainData.drop("accident_risk", axis=1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=999)
X_train.head(10)

In [ ]:
#Preprocesssing Pipelines
ordinal_features = ["road_type", "lighting", "weather", "time_of_day"]
numeric_features = ["num_lanes", "curvature", "speed_limit", "num_reported_accidents"]
boolean_features = ["road_signs_present", "public_road", "holiday", "school_season"]
categorical_features = ordinal_features
ord_enc_v1 = OrdinalEncoder(categories=[["rural", "highway", "urban"],
                                       ["daylight", "dim", "night"],
                                       ["clear","foggy","rainy"],
                                       ["morning", "afternoon", "evening"]])

ord_enc_v2 = OrdinalEncoder(categories=[["urban", "highway", "rural"],
                                       ["daylight", "night", "dim"],
                                       ["clear","foggy","rainy"],
                                       ["morning", "afternoon", "evening"]])
trainDatav1 = trainData
traindatav2 = trainData
ordinal_pipeline_v1 = Pipeline([
    ('ordinal_encoder', ord_enc_v1)
])
ordinal_pipeline_v2 = Pipeline([
    ('ordinal_encoder', ord_enc_v2)
])
numeric_pipeline = Pipeline([
    ('scaler', StandardScaler())
])
boolean_pipeline = Pipeline([
    ('bool_encoder', FunctionTransformer(lambda x: x.astype(int)))
])
preprocessor_v1 = ColumnTransformer(transformers=[
    ('ord', ordinal_pipeline_v1, ordinal_features),
    ('num', numeric_pipeline, numeric_features),
    ('bool', boolean_pipeline, boolean_features)
    ],remainder='passthrough')
preprocessor_v2 = ColumnTransformer(transformers=[
    ('ord', ordinal_pipeline_v2, ordinal_features),
    ('num', numeric_pipeline, numeric_features),
    ('bool', boolean_pipeline, boolean_features)
    ],remainder='passthrough')

In [ ]:
#Preproccessing pipeline usage
X_trainProcessedV1 = preprocessor_v1.fit_transform(X_train)
X_testProcessedV1 = preprocessor_v1.fit_transform(X_test)
X_trainProcessedV2 = preprocessor_v2.fit_transform(X_train)
X_testProcessedV2 = preprocessor_v2.fit_transform(X_test)
CompProcessed = preprocessor_v1.fit_transform(CompData)

In [ ]:
#Objective methods for different models and data versions from preprossesors 
def objectiveElasticNet_PV1(trail):
    params = {
        'alpha': trail.suggest_float('alpha', 0.01, 1.0),
        'l1_ratio': trail.suggest_float('l1_ratio', 0.01, 0.1),
        'max_iter': trail.suggest_int('max_iter', 950, 1100),
        'tol': trail.suggest_float('tol', 0.000000001, 0.00001),
        'precompute': False,
        'positive': False,
        'warm_start': False,
        'selection': trail.suggest_categorical('selection', ['cyclic', 'random'])
    }

    model = ElasticNet(**params, random_state=999)
    model.fit(X_trainProcessedV1, y_train)
    preds = model.predict(X_testProcessedV1)
    error = root_mean_squared_error(y_test, preds)#SQRT OCCURS HERE DO NOT ROOT TWICE!
    return error

def objectiveElasticNet_PV2(trail):
    params = {
        'alpha': trail.suggest_float('alpha', 0.01, 1.0),
        'l1_ratio': trail.suggest_float('l1_ratio', 0.0, 1),
        'max_iter': trail.suggest_int('max_iter', 500, 2500),
        'tol': trail.suggest_float('tol', 0.00001, 0.001),
    }

    model = ElasticNet(**params, random_state=999, precompute = False, positive = False, warm_start = False, selection = 'random')
    model.fit(X_trainProcessedV2, y_train)
    preds = model.predict(X_testProcessedV2)
    error = root_mean_squared_error(y_test, preds)#SQRT OCCURS HERE DO NOT ROOT TWICE!
    return error

def objectiveXGBoost_PV1(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.003),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 0.1),
        'reg_alpha': trial.suggest_float('reg_alpha', 1, 50),
        'reg_lambda': trial.suggest_float('reg_lambda', 1, 50),
        'min_child_weight': trial.suggest_float('min_child_weight', 1, 10),
        'max_delta_step' : trial.suggest_int('max_delta_step', 1, 20),
    }

    model = XGBRegressor(**params, random_state=999, objective = 'reg:squarederror')
    model.fit(X_trainProcessedV1, y_train)
    preds = model.predict(X_testProcessedV1)
    error = root_mean_squared_error(y_test, preds)#SQRT OCCURS HERE DO NOT ROOT TWICE!
    return error

def singletuneXGBoost_PV1(trial):
    pass
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.003),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 0.1),
        'reg_alpha': trial.suggest_float('reg_alpha', 1, 50),
        'reg_lambda': trial.suggest_float('reg_lambda', 1, 50),
        'min_child_weight': trial.suggest_float('min_child_weight', 1, 10),
        'max_delta_step' : trial.suggest_int('max_delta_step', 1, 20),
    }

    model = XGBRegressor(**params, random_state=999, objective = 'reg:squarederror')
    model.fit(X_trainProcessedV1, y_train)
    preds = model.predict(X_testProcessedV1)
    error = root_mean_squared_error(y_test, preds)#SQRT OCCURS HERE DO NOT ROOT TWICE!
    return error

def SVMTune(trial):
    kernel = trial.suggest_categorical('kernel', ['linear', 'poly', 'rbf', 'sigmoid'])
    params = {
    'kernel': kernel,
    'C': trial.suggest_float('C', 1e-3, 1e3, log=True),
    'gamma': trial.suggest_float('gamma', 1e-4, 1e1, log=True) if kernel in ['rbf', 'poly', 'sigmoid'] else 'scale',
    'degree': trial.suggest_int('degree', 2, 5) if kernel == 'poly' else 3,
    'coef0': trial.suggest_float('coef0', 0.0, 1.0) if kernel in ['poly', 'sigmoid'] else 0.0,
    }

    choice = trial.suggest_categorical('XT', ['v1', 'v2'])
    X_TRN = X_trainProcessedV1 if choice == 'v1' else X_trainProcessedV2
    X_TST = X_testProcessedV1 if choice == 'v1' else X_testProcessedV2
    model = SVR()
    model.fit(X_TRN, y_train)
    preds = model.predict(X_TST)
    error = root_mean_squared_error(y_test, preds)#SQRT OCCURS HERE DO NOT ROOT TWICE!
    return error

    


    
    #Below are optuna studies, one for each optimization method.

In [ ]:
ElasticNet_PV1 = optuna.create_study(study_name="ElasticNet_PV1", storage="sqlite:///optuna_study.db", sampler=optuna.samplers.TPESampler(seed=999), pruner=optuna.pruners.MedianPruner(n_startup_trials=15, n_warmup_steps=5, interval_steps=1, n_min_trials=50), direction='minimize', load_if_exists=True)
ElasticNet_PV1.optimize(objectiveElasticNet_PV1, n_trials=10, show_progress_bar=True)

In [ ]:
ElasticNet_PV1_NP = optuna.create_study(study_name="ElasticNet_PV1_NP", storage="sqlite:///optuna_study.db", sampler=optuna.samplers.TPESampler(seed=999), pruner=optuna.pruners.MedianPruner(n_startup_trials=15, n_warmup_steps=5, interval_steps=1, n_min_trials=50), direction='minimize', load_if_exists=True)
ElasticNet_PV1_NP.optimize(objectiveElasticNet_PV1, n_trials=10, show_progress_bar=True)

In [ ]:
SupportVectorStudy = optuna.create_study(study_name="SVM_PV1", storage="sqlite:///optuna_study.db", sampler=optuna.samplers.TPESampler(seed=999), pruner=optuna.pruners.MedianPruner(n_startup_trials=15, n_warmup_steps=5, interval_steps=1, n_min_trials=50), direction='minimize', load_if_exists=True)
SupportVectorStudy.optimize(SVMTune, n_trials = 10, show_progress_bar=True)

In [ ]:
ENPV1_Best_Params = ElasticNet_PV1.best_params

In [ ]:
#Creates a submission file for competition submission
def makeCompSubmission(BEST_PARAMS, filename, modelname):
    model = modelname(**BEST_PARAMS)
    model.fit(X_trainProcessedV1, y_train)
    preds = model.predict(CompProcessed)
    submission_df = pd.DataFrame({"id": range(517754,517754 + len(preds)),"accident_risk": preds})
    submission_df.to_csv(filename+'.csv', index=False)
    print("OutputSize: " + str(len(preds)))
    print('Best Params: '+ str(BEST_PARAMS))

In [ ]:
makeCompSubmission(ENPV1_Best_Params, "submissionv1", ElasticNet)